[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C01_LLM_Internals_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检 Environment Check

> **MODULE 00 / 8** · 配套讲解：[`00_overview.html`](00_overview.html) · 全程 **纯 CPU**，无需 GPU、无需任何 API key。

这本 notebook 做四件事，全部通过即环境就绪：

1. **版本自检**：检查全部依赖是否安装（缺失的会标红）；
2. **device 检测**：本课 CPU 即可，有 GPU / Apple MPS 加速更好；
3. **PyTorch 冒烟测试**：①矩阵乘 $(2,3)\times(3,4)\to(2,4)$，数值与手算对照；
   ②autograd——对 $y=\sum_i x_i^2$，`backward()` 的梯度和解析解 $2x_i$ 对照，再用有限差分交叉验证；
4. **两道热身 ✏️ 练习**：数值稳定 softmax 与单样本 cross-entropy —— 它们是全课最常用的两个算子，
   也是你第一次体验本课的核心方法论：**亲手实现 → 与官方实现数值对拍（assert）**。

⚠️ 运行前确认右上角 kernel 是 **LLM Internals Course**（环境配置见 `00_overview.html` 第 4 节）。

In [ ]:
# === 1. 版本自检：缺失的依赖标红 ===
import sys
import importlib

print(f"Python {sys.version.split()[0]}  ({sys.executable})")
assert sys.version_info >= (3, 10), "本课需要 Python >= 3.10"

RED, GREEN, RESET = "\033[31m", "\033[32m", "\033[0m"
required = ["torch", "numpy", "scipy", "matplotlib", "tiktoken", "transformers"]

missing = []
for name in required:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, "__version__", "?")
        print(f"{GREEN}  ok  {name:<14}{ver}{RESET}")
    except ImportError:
        missing.append(name)
        print(f"{RED}  缺失  {name:<14}未安装{RESET}")

if missing:
    print(f"\n{RED}缺少 {len(missing)} 个依赖，请在终端运行：")
    print(f"  pip install {' '.join(missing)}{RESET}")
else:
    print("\n✅ 全部依赖就绪")

# === 2. device 检测：本课 CPU 即可，GPU/MPS 是加速可选项 ===
import torch

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
    print("检测到 CUDA GPU:", torch.cuda.get_device_name(0))
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = "mps"
    print("检测到 Apple Silicon GPU (MPS)")
else:
    print("未检测到 GPU —— 完全没问题！")

print(f"\ndevice = '{device}'")
print("全课 8 个模块按 CPU 设计：mini 规模 GPT 在 CPU 上几分钟训完。")
print("有 GPU/MPS 时部分模块会自动加速，但所有结论与练习不依赖它。")

In [ ]:
# === 3. PyTorch 冒烟测试：全部用 assert 与手算结果对照（"数值对拍"的最小示范） ===
import torch

# --- 3a. 矩阵乘：形状 + 数值与手算对照 ---
A = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])          # 形状 (2, 3)
B = torch.ones(3, 4)                      # 形状 (3, 4)
C = A @ B                                 # (2,3) @ (3,4) -> (2,4)
assert C.shape == (2, 4), f"形状不对: {C.shape}"
# 手算：B 全 1，所以 C 每列都是 A 的行和 -> [1+2+3, 4+5+6] = [6, 15]
assert torch.allclose(C[:, 0], torch.tensor([6., 15.]))
print("矩阵乘 OK:", tuple(C.shape), "| 第一列 =", C[:, 0].tolist())

# --- 3b. autograd：backward() 与手算解析梯度对照 ---
x = torch.tensor([2., 3.], requires_grad=True)
y = (x ** 2).sum()            # y = x1^2 + x2^2
y.backward()                  # 解析梯度: dy/dx_i = 2*x_i -> [4, 6]
assert torch.allclose(x.grad, torch.tensor([4., 6.]))
print("autograd 与手算一致: grad =", x.grad.tolist())

# --- 3c. 有限差分数值梯度交叉验证: (f(x+eps) - f(x-eps)) / (2*eps) ---
# 注意：数值梯度要用 float64 算 —— float32 下 eps=1e-4 的舍入误差会淹没结果
def f(v):
    return (v ** 2).sum()

xd = x.detach().double()
eps = 1e-4
num_grad = torch.zeros(2, dtype=torch.float64)
for i in range(2):
    e = torch.zeros(2, dtype=torch.float64)
    e[i] = eps
    num_grad[i] = (f(xd + e) - f(xd - e)) / (2 * eps)
assert torch.allclose(num_grad, x.grad.double(), atol=1e-6)
print("数值梯度交叉验证 OK:", [round(g, 4) for g in num_grad.tolist()])

print("\n✅ PyTorch 冒烟测试全部通过")

## ✏️ 练习 1：实现数值稳定版 `softmax(x)`

$$\mathrm{softmax}(x)_i = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

**任务**：对 1-D 张量实现 softmax。难点在数值稳定：当 $x$ 里有大数（如 1000）时，
`exp(1000)` 在 float32 下直接溢出为 `inf`。

**提示**：利用恒等式 $\mathrm{softmax}(x) = \mathrm{softmax}(x - c)$（分子分母同乘 $e^{-c}$），
取 $c = \max(x)$，则 $x - c \le 0$，`exp` 永不溢出。3 行代码即可完成。

（softmax 是 attention 与解码采样的核心算子，模块 02/04 会反复用到。）

In [ ]:
import torch

def softmax(x: torch.Tensor) -> torch.Tensor:
    '''数值稳定版 softmax（输入 1-D 张量，沿最后一维归一化）。

    步骤：
      1. m = x 的最大值（torch.max / x.max()）
      2. e = torch.exp(x - m)    # 减 max：指数永不溢出
      3. 返回 e / e.sum()
    '''
    # TODO: 在这里实现（约 3 行），然后删掉下面这行
    raise NotImplementedError

In [ ]:
# === 练习 1 自测：与 torch.softmax 数值对拍 ===
torch.manual_seed(0)

# 用例 1：普通随机向量，与官方实现对拍（atol=1e-6）
v = torch.randn(10)
assert torch.allclose(softmax(v), torch.softmax(v, dim=-1), atol=1e-6), \
    "普通输入与 torch.softmax 不一致"

# 用例 2：大数不溢出（不减 max 的话 exp(1000) = inf）
big = torch.tensor([1000., 1001., 1002.])
out = softmax(big)
assert torch.isfinite(out).all(), "出现 inf/nan：确认你减了 max"
assert torch.allclose(out, torch.softmax(big, dim=-1), atol=1e-6)

# 用例 3：极小的数同样稳定
neg = torch.tensor([-1000., -1001., -999.])
assert torch.isfinite(softmax(neg)).all()

# 用例 4：输出是合法概率分布（和为 1）
assert abs(softmax(v).sum().item() - 1.0) < 1e-6

print("✅ 练习 1 通过")

## ✏️ 练习 2：实现单样本 `cross_entropy(logits, target_idx)`

语言模型的训练目标。对 logits $z \in \mathbb{R}^V$ 与真实类别 $t$：

$$\mathrm{CE}(z, t) = -\log\mathrm{softmax}(z)_t = -z_t + \underbrace{\log\sum_j e^{z_j}}_{\text{log-sum-exp}}$$

**任务**：实现上式。直接算 $\log\sum e^{z_j}$ 会在大 logits 下溢出，必须用
<b>log-sum-exp 技巧</b>：

$$\log\sum_j e^{z_j} = m + \log\sum_j e^{z_j - m}, \qquad m = \max_j z_j$$

**提示**：3 行——先算 `m`，再算 `lse = m + torch.log(torch.exp(z - m).sum())`，
最后返回 `lse - z[target_idx]`。

**自检直觉**：当 logits 全相等（均匀分布）时，$\mathrm{CE} = \ln V$ ——
这正是模块 03 里判断"模型有没有开始学习"的 baseline。

In [ ]:
import torch

def cross_entropy(logits: torch.Tensor, target_idx: int) -> torch.Tensor:
    '''单样本交叉熵：CE = -logits[t] + logsumexp(logits)，用 log-sum-exp 技巧保证稳定。

    参数：
      logits     : 1-D 张量，形状 (V,)，未归一化分数
      target_idx : int，真实类别下标
    返回：标量张量
    '''
    # TODO: m = logits 的最大值
    # TODO: lse = m + log(sum(exp(logits - m)))
    # TODO: 返回 lse - logits[target_idx]
    raise NotImplementedError

In [ ]:
# === 练习 2 自测：与 F.cross_entropy 数值对拍 ===
import math
import torch.nn.functional as F

torch.manual_seed(1)

# 用例 1：随机 logits，与官方实现对拍（官方接口吃 batch，这里 unsqueeze 成 batch=1）
z = torch.randn(7)
t = 3
ref = F.cross_entropy(z.unsqueeze(0), torch.tensor([t]))
assert torch.allclose(cross_entropy(z, t), ref, atol=1e-6), \
    "普通输入与 F.cross_entropy 不一致"

# 用例 2：大 logits 不溢出（不用 log-sum-exp 的话 exp(1010) = inf）
big = torch.tensor([1000., 1010., 990.])
got = cross_entropy(big, 0)
ref_big = F.cross_entropy(big.unsqueeze(0), torch.tensor([0]))
assert torch.isfinite(got), "出现 inf/nan：用 log-sum-exp 技巧"
assert torch.allclose(got, ref_big, atol=1e-4)

# 用例 3：均匀 logits -> CE = ln(V)（模块 03 的训练 baseline）
V = 50
uniform = torch.zeros(V)
assert abs(cross_entropy(uniform, 0).item() - math.log(V)) < 1e-6

print("✅ 练习 2 通过")

---

## 📖 参考答案

**先自己做，再对照。** 卡住超过 20 分钟可以看；看完后建议把答案 cell 复制回练习 cell 重新跑一遍自测。

In [ ]:
# 参考答案 · 练习 1（先自己做，再对照）
import torch

def softmax(x: torch.Tensor) -> torch.Tensor:
    m = x.max()                  # 1. 取最大值
    e = torch.exp(x - m)         # 2. 平移后再取指数：x - m <= 0，永不溢出
    return e / e.sum()           # 3. 归一化

# 减 max 不改变结果：softmax(x) = softmax(x - c)，因为分子分母同乘 e^{-c} 约掉
v = torch.randn(10)
assert torch.allclose(softmax(v), torch.softmax(v, dim=-1), atol=1e-6)
assert torch.isfinite(softmax(torch.tensor([1000., 1001., 1002.]))).all()
print("参考答案 1 验证通过")

In [ ]:
# 参考答案 · 练习 2（先自己做，再对照）
import math
import torch
import torch.nn.functional as F

def cross_entropy(logits: torch.Tensor, target_idx: int) -> torch.Tensor:
    m = logits.max()                                   # log-sum-exp 技巧第 1 步
    lse = m + torch.log(torch.exp(logits - m).sum())   # logsumexp(z) = m + log(sum(exp(z-m)))
    return lse - logits[target_idx]                    # CE = lse - z_t

z = torch.randn(7)
ref = F.cross_entropy(z.unsqueeze(0), torch.tensor([3]))
assert torch.allclose(cross_entropy(z, 3), ref, atol=1e-6)
assert torch.isfinite(cross_entropy(torch.tensor([1000., 1010., 990.]), 0))
assert abs(cross_entropy(torch.zeros(50), 0).item() - math.log(50)) < 1e-6
print("参考答案 2 验证通过")

---

## 每模块算力一览（全部 CPU 可跑）

| # | 模块 | Notebook | 算力 | 预计运行时间 |
|---|------|----------|------|--------------|
| 00 | 课程总览与环境 | `00_environment_check.ipynb` | CPU | < 1 分钟 |
| 01 | Tokenization：从零实现 BPE | `01_bpe_from_scratch.ipynb` | CPU | 1–2 分钟 |
| 02 | 手写 Attention 与 Transformer Block | `02_attention_transformer.ipynb` | CPU | 1–2 分钟 |
| 03 | 从零训练 mini-GPT | `03_train_minigpt.ipynb` | CPU | 训练约 3–5 分钟 |
| 04 | 解码策略全手写 | `04_decoding_strategies.ipynb` | CPU | 1–2 分钟 |
| 05 | Scaling Laws：亲手拟合 | `05_scaling_laws.ipynb` | CPU | 5–10 分钟（训一组小模型） |
| 06 | KV Cache 与高效推理 | `06_kv_cache.ipynb` | CPU | 1–2 分钟 |
| 07 | 现代架构：RoPE/GQA/MoE/SSM | `07_modern_architectures.ipynb` | CPU | 2–3 分钟 |

GPU / Apple MPS 会让 03/05 更快，但不是必需；全课不下载大模型权重、不调用任何外部 API。

## 下一站 → 模块 01

环境就绪、两道练习的 ✅ 都拿到了？前往 **[01 · Tokenization：从零实现 BPE](../01_tokenization/01_讲解.html)**：
亲手实现 BPE 训练与编码/解码，并与 `tiktoken` 逐 token 对拍 —— 那是"字符 → token → 张量 → loss → 生成"主线的第一步。